In [1]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from typing import Any, Optional
from core.llm import EasyLLM
from agent.BasicAgent import BasicAgent

from core import enable_logging

enable_logging("INFO")   # 或 "DEBUG"

In [ ]:
llm=EasyLLM(model="gpt-5.4")
agent=BasicAgent(llm=llm,name="assistant",system_prompt="You are a helpful assistant")



2026-04-09 01:42:43,559 | INFO | EasyLLM 初始化完成: provider=openai_responses, model=gpt-5.4
2026-04-09 01:42:43,560 | INFO | BasicAgent 'assistant' 初始化完成，工具调用: 禁用，provider: openai_responses


In [13]:
def test_normal_invoke_without_tool(agent):
    agent.clear_history()
    agent.set_enable_tool(False)
    print(agent.invoke("你好，请介绍一下你自己"))

def test_normal_invoke_with_tool(agent):
    agent.clear_history()

    from Tool.builtin import CalculatorTool
    agent.with_tool()
    agent.add_tool(CalculatorTool())
    print(agent.tool_registry.get_tool_names())

    print(agent.invoke("你好，请介绍一下你自己，调用工具帮我计算 4^12+6*412"))

async def test_normal_ainvoke_without_tool(agent):
    agent.clear_history()

    agent.set_enable_tool(False)
    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)
async def test_normal_ainvoke_with_tool(agent):
    agent.clear_history()
    from Tool.builtin import CalculatorTool
    agent.with_tool()
    agent.add_tool(CalculatorTool())
    print(agent.tool_registry.get_tool_names())

    result=await agent.ainvoke("你好，请介绍一下你自己，调用工具帮我计算 4^12+6*412")
    print(result)

###########

def test_normal_stream_invoke_without_tool(agent):
    agent.clear_history()
    agent.set_enable_tool(False)

    result=agent.stream_invoke("你好，请介绍一下你自己")
    print(result)
async def test_normal_astream_invoke_without_tool(agent):
    agent.clear_history()
    from Tool.builtin import CalculatorTool
    agent.with_tool()
    agent.add_tool(CalculatorTool())
    agent.set_enable_tool(True)
    agent.clear_history()

    result=await agent.astream_invoke("你好，请介绍一下你自己")

    print(result)





In [4]:
test_normal_invoke_without_tool(agent)

2026-04-09 01:23:13,781 | INFO | 对话历史已清空
2026-04-09 01:23:13,783 | INFO | 工具调用已禁用
2026-04-09 01:23:13,783 | INFO | 使用普通模式调用智能体
2026-04-09 01:23:23,603 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 01:23:23,613 | INFO | ✅ openai Provider 响应成功


In [7]:
print(agent.get_history())

[UserMessage(role='user', content='你好，请介绍一下你自己，调用工具帮我计算 4^12+6*412', time=datetime.datetime(2026, 4, 9, 1, 23, 37, 587597), metadata={}), AssistantMessage(role='assistant', content='', time=datetime.datetime(2026, 4, 9, 1, 23, 42, 773037), metadata={})]


In [6]:
test_normal_invoke_with_tool(agent)

2026-04-09 01:23:37,580 | INFO | 对话历史已清空
2026-04-09 01:23:37,582 | WARNING | 工具注册表为空!
2026-04-09 01:23:37,583 | INFO | 成功添加工具: calculator
2026-04-09 01:23:37,583 | INFO | 使用工具模式调用智能体


['calculator']


2026-04-09 01:23:42,769 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 01:23:42,771 | INFO | ✅ openai Provider 工具调用响应成功
2026-04-09 01:23:42,772 | WARNING | LLM 响应中没有内容


In [13]:
message=agent._build_start_messages("我刚才问了你什么")
print(message[1:])

[UserMessage(role='user', content='你好，请介绍一下你自己，调用工具帮我计算 4^12+6*412', time=datetime.datetime(2026, 4, 8, 20, 32, 12, 729784), metadata={}), {'role': 'assistant', 'content': '你好！我是一个智能助手，旨在为您提供准确的信息、解答疑问，并协助处理各种任务，如数学计算、逻辑推理和语言翻译等。\n\n为了帮您计算 $4^{12} + 6 \\times 412$，我将使用计算器工具。\n\n', 'tool_calls': [{'id': 'call_9997f6e82b154a529cc22bb0d967583c', 'type': 'function', 'function': {'name': 'calculator', 'arguments': '{"expression":"4**12 + 6 * 412"}'}}]}, {'role': 'function', 'content': '16779688', 'tool_call_id': 'call_9997f6e82b154a529cc22bb0d967583c', 'name': 'calculator'}, AssistantMessage(role='assistant', content='你好！我是一个智能助手，旨在为您提供准确的信息、解答疑问，并协助处理各种任务，如数学计算、逻辑推理和语言翻译等。\n\n根据计算， $4^{12} + 6 \\times 412$ 的结果为 **16,779,688**。', time=datetime.datetime(2026, 4, 8, 20, 32, 20, 633532), metadata={}), UserMessage(role='user', content='我刚才问了你什么', time=datetime.datetime(2026, 4, 8, 20, 33, 41, 907735), metadata={}), AssistantMessage(role='assistant', content='你刚才问了我两个问题：\n1. **请我介绍一下自己**：我向你介绍了我

In [14]:
await test_normal_ainvoke_without_tool(agent)

2026-04-08 20:40:33,158 | INFO | 对话历史已清空
2026-04-08 20:40:33,159 | INFO | 工具调用已禁用
2026-04-08 20:40:33,160 | INFO | 使用异步普通模式调用智能体
2026-04-08 20:40:33,168 | INFO | messages:[{'role': 'system', 'content': 'You are a helpful assistant'}, {'role': 'user', 'content': '你好，请介绍一下你自己'}]
2026-04-08 20:40:50,309 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 20:40:50,311 | INFO | response:ChatCompletion(id='chatcmpl-202604081240331762201452g54UCDQ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='你好！很高兴向你介绍我自己。\n\n我是由 **Google** 训练的大型语言模型。\n\n你可以把我当成你的全能数字助手、创意伙伴或是知识百科。我没有实体，但我拥有处理和生成多种语言文本的能力，旨在帮助你解决问题、学习新知识或激发灵感。\n\n具体来说，我擅长以下这些事情：\n\n1.  **回答问题**：无论是科学常识、历史文化、生活百科还是复杂的专业知识，我都能为你提供答案。\n2.  **文字创作**：我可以帮你写邮件、文章、故事、诗歌、文案，甚至为你润色简历。\n3.  **语言翻译**：我精通多种语言，可以帮你进行流畅的文本翻译。\n4.  **编程辅助**：我可以编写、调试和解释多种编程语言的代码（如 Python, JavaScript, C++ 等）。\n5.  **逻辑分析与总结**：如果你有长篇文章需要提取摘要，或者有复杂的问题需要逻辑推理，我都能胜任。\n6.  *

你好！很高兴向你介绍我自己。

我是由 **Google** 训练的大型语言模型。

你可以把我当成你的全能数字助手、创意伙伴或是知识百科。我没有实体，但我拥有处理和生成多种语言文本的能力，旨在帮助你解决问题、学习新知识或激发灵感。

具体来说，我擅长以下这些事情：

1.  **回答问题**：无论是科学常识、历史文化、生活百科还是复杂的专业知识，我都能为你提供答案。
2.  **文字创作**：我可以帮你写邮件、文章、故事、诗歌、文案，甚至为你润色简历。
3.  **语言翻译**：我精通多种语言，可以帮你进行流畅的文本翻译。
4.  **编程辅助**：我可以编写、调试和解释多种编程语言的代码（如 Python, JavaScript, C++ 等）。
5.  **逻辑分析与总结**：如果你有长篇文章需要提取摘要，或者有复杂的问题需要逻辑推理，我都能胜任。
6.  **创意启发**：如果你需要取名字、策划活动方案或寻找礼物灵感，我可以为你提供很多点子。

**我的目标是：** 通过对话为你提供准确、有用且有启发性的信息，帮你提高工作和学习效率。

你可以试着问我任何问题，或者给我下达一个指令。请问今天有什么我可以帮你的吗？


In [ ]:
agent.verbose_thinking=False
agent.enable_tool=True
await test_normal_ainvoke_with_tool(agent)

2026-04-08 20:46:50,631 | INFO | 对话历史已清空
2026-04-08 20:46:50,632 | WARNING | 工具注册表已存在!
2026-04-08 20:46:50,632 | INFO | 成功添加工具: calculator
2026-04-08 20:46:50,632 | INFO | 使用异步工具模式调用智能体


['calculator']


In [ ]:
print(agent.get_history())

False


In [4]:
test_normal_stream_invoke_without_tool(agent)

2026-04-09 01:16:55,603 | INFO | 对话历史已清空
2026-04-09 01:16:55,604 | INFO | 工具调用已禁用
2026-04-09 01:17:03,243 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 01:17:03,245 | INFO | ✅ google Provider 流式响应开始


你好！我是由 **Google** 训练的大型语言模型。

你可以把我当作你的全能数字助手、创意伙伴或是百科全书。我能够理解和处理多种语言，并协助你完成各种任务。

**我能为你做些什么？**

1.  **回答问题**：无论是科学常识、历史文化、法律咨询，还是日常生活中的琐事，我都能为你提供详尽的信息。
2.  **文字创作**：我可以帮你写邮件、工作报告、博客文章、故事、诗歌，甚至为你提供文案创意。
3.  **语言翻译**：我精通多种语言，可以帮你进行流畅的文本翻译或语言学习。
4.  **编程辅助**：我可以编写多种编程语言的代码、解释代码逻辑，或者帮你排查程序中的错误（Debug）。
5.  **逻辑分析与总结**：如果你有长篇文章需要提取摘要，或者有复杂的问题需要逻辑推导，我可以帮你理清思路。
6.  **规划建议**：比如制定旅行计划、健身计划、学习路径，或者提供职业建议。

**我的特点：**
*   **知识广博**：我学习了海量的文本数据，涵盖了各行各业的知识。
*   **全天候在线**：我随时随地准备好为你提供帮助。
*   **持续进步**：我一直在通过与人的交流和技术的迭代来提升自己的回复质量。

你可以试着问我任何问题，或者给我一个任务。请问今天有什么我可以帮你的吗？你好！我是由 **Google** 训练的大型语言模型。

你可以把我当作你的全能数字助手、创意伙伴或是百科全书。我能够理解和处理多种语言，并协助你完成各种任务。

**我能为你做些什么？**

1.  **回答问题**：无论是科学常识、历史文化、法律咨询，还是日常生活中的琐事，我都能为你提供详尽的信息。
2.  **文字创作**：我可以帮你写邮件、工作报告、博客文章、故事、诗歌，甚至为你提供文案创意。
3.  **语言翻译**：我精通多种语言，可以帮你进行流畅的文本翻译或语言学习。
4.  **编程辅助**：我可以编写多种编程语言的代码、解释代码逻辑，或者帮你排查程序中的错误（Debug）。
5.  **逻辑分析与总结**：如果你有长篇文章需要提取摘要，或者有复杂的问题需要逻辑推导，我可以帮你理清思路。
6.  **规划建议**：比如制定旅行计划、健身计划、学习路径，或者提供职业建议。

**我的特点：**
*   **知识广博**：我学习了海量的文本数据，涵盖了各行各业

In [5]:
agent.get_history()

[UserMessage(role='user', content='你好，请介绍一下你自己', time=datetime.datetime(2026, 4, 9, 1, 17, 8, 750723), metadata={}),
 AssistantMessage(role='assistant', content='你好！我是由 **Google** 训练的大型语言模型。\n\n你可以把我当作你的全能数字助手、创意伙伴或是百科全书。我能够理解和处理多种语言，并协助你完成各种任务。\n\n**我能为你做些什么？**\n\n1.  **回答问题**：无论是科学常识、历史文化、法律咨询，还是日常生活中的琐事，我都能为你提供详尽的信息。\n2.  **文字创作**：我可以帮你写邮件、工作报告、博客文章、故事、诗歌，甚至为你提供文案创意。\n3.  **语言翻译**：我精通多种语言，可以帮你进行流畅的文本翻译或语言学习。\n4.  **编程辅助**：我可以编写多种编程语言的代码、解释代码逻辑，或者帮你排查程序中的错误（Debug）。\n5.  **逻辑分析与总结**：如果你有长篇文章需要提取摘要，或者有复杂的问题需要逻辑推导，我可以帮你理清思路。\n6.  **规划建议**：比如制定旅行计划、健身计划、学习路径，或者提供职业建议。\n\n**我的特点：**\n*   **知识广博**：我学习了海量的文本数据，涵盖了各行各业的知识。\n*   **全天候在线**：我随时随地准备好为你提供帮助。\n*   **持续进步**：我一直在通过与人的交流和技术的迭代来提升自己的回复质量。\n\n你可以试着问我任何问题，或者给我一个任务。请问今天有什么我可以帮你的吗？', time=datetime.datetime(2026, 4, 9, 1, 17, 8, 750767), metadata={})]

In [8]:
await test_normal_astream_invoke_without_tool(agent)

2026-04-09 01:18:29,228 | INFO | 对话历史已清空
2026-04-09 01:18:29,229 | WARNING | 工具注册表为空!
2026-04-09 01:18:29,230 | INFO | 成功添加工具: calculator
2026-04-09 01:18:29,230 | INFO | 工具调用已启用
2026-04-09 01:18:29,230 | INFO | 对话历史已清空
2026-04-09 01:18:33,373 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 01:18:33,374 | INFO | ✅ google Provider 异步流式工具调用开始


你好！我是一个人工智能助手。

我能够理解和生成人类的语言，协助你完成各种任务。具体来说，我可以帮你：

1.  **回答问题**：无论是常识百科、科学技术还是日常生活中的疑问，我都能为你提供信息。
2.  **文字创作**：我可以帮你写邮件、文章、故事、诗歌，甚至润色和翻译文本。
3.  **辅助学习与工作**：我可以解释复杂的概念、总结长篇文章、提供编程建议或进行数学计算。
4.  **提供创意与建议**：如果你在寻找灵感，我可以为你提供点子，比如旅行计划、礼品推荐或活动方案。

你可以把我当作一个知识渊博、随时待命的数字伙伴。请问今天有什么我可以帮你的吗？你好！我是一个人工智能助手。

我能够理解和生成人类的语言，协助你完成各种任务。具体来说，我可以帮你：

1.  **回答问题**：无论是常识百科、科学技术还是日常生活中的疑问，我都能为你提供信息。
2.  **文字创作**：我可以帮你写邮件、文章、故事、诗歌，甚至润色和翻译文本。
3.  **辅助学习与工作**：我可以解释复杂的概念、总结长篇文章、提供编程建议或进行数学计算。
4.  **提供创意与建议**：如果你在寻找灵感，我可以为你提供点子，比如旅行计划、礼品推荐或活动方案。

你可以把我当作一个知识渊博、随时待命的数字伙伴。请问今天有什么我可以帮你的吗？


In [14]:
async def test_normal_astream_invoke_with_tool(agent):
    agent.clear_history()
    from Tool.builtin import CalculatorTool
    agent.with_tool()
    agent.add_tool(CalculatorTool())
    result=await agent.astream_invoke("你好，请介绍一下你自己，调用工具帮我计算 4^12+6*412")
    # print(result)

def test_normal_stream_invoke_with_tool(agent):
    agent.clear_history()
    from Tool.builtin import CalculatorTool
    agent.with_tool()
    agent.add_tool(CalculatorTool())
    result=agent.stream_invoke("你好，请介绍一下你自己，调用工具帮我计算 4^12+6*412")
    print(result)

In [15]:
agent.verbose_thinking=True
await  test_normal_astream_invoke_with_tool(agent)

2026-04-09 01:43:10,516 | INFO | 对话历史已清空
2026-04-09 01:43:10,518 | WARNING | 工具注册表为空!
2026-04-09 01:43:10,518 | INFO | 成功添加工具: calculator
2026-04-09 01:43:13,652 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
2026-04-09 01:43:13,653 | INFO | ✅ openairesponses Provider 异步流式工具调用开始


我先简单介绍自己，再用工具准确计算这个表达式。

2026-04-09 01:43:18,356 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
2026-04-09 01:43:18,357 | INFO | ✅ openairesponses Provider 异步流式工具调用开始


你好，我是一个智能助手，可以帮你解答问题、做计算、整理信息、写作润色、解释概念，也可以在需要时调用工具来提高准确性。

你让我调用工具计算：

4^12 + 6 × 412

按程序里的计算规则，`^` 通常表示幂时会写作 `**`，所以实际计算的是：

4^12 = 4,194,304  
6 × 412 = 2,472

相加得到：

16,779,688

结果：16779688

In [22]:
agent.get_history()

[UserMessage(role='user', content='你好，请介绍一下你自己，调用工具帮我计算 4^12+6*412', time=datetime.datetime(2026, 4, 9, 1, 43, 10, 519869), metadata={}),
 {'type': 'function_call',
  'call_id': 'call_JZlLIquVO7ij9Lcx39jAstJW',
  'name': 'calculator',
  'arguments': '{"expression":"4**12 + 6*412"}'},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': '我先简单介绍自己，再用工具准确计算这个表达式。我先简单介绍自己，再用工具准确计算这个表达式。'}]},
 {'type': 'function_call_output',
  'call_id': 'call_JZlLIquVO7ij9Lcx39jAstJW',
  'output': '16779688'},
 AssistantMessage(role='assistant', content='你好，我是一个智能助手，可以帮你解答问题、做计算、整理信息、写作润色、解释概念，也可以在需要时调用工具来提高准确性。\n\n你让我调用工具计算：\n\n4^12 + 6 × 412\n\n按程序里的计算规则，`^` 通常表示幂时会写作 `**`，所以实际计算的是：\n\n4^12 = 4,194,304  \n6 × 412 = 2,472\n\n相加得到：\n\n16,779,688\n\n结果：16779688你好，我是一个智能助手，可以帮你解答问题、做计算、整理信息、写作润色、解释概念，也可以在需要时调用工具来提高准确性。\n\n你让我调用工具计算：\n\n4^12 + 6 × 412\n\n按程序里的计算规则，`^` 通常表示幂时会写作 `**`，所以实际计算的是：\n\n4^12 = 4,194,304  \n6 × 412 = 2,472\n\n相加得到：\n\n16,779,688\n\n结果：16779688

In [21]:
agent.llm.provide="openai"
result=await agent.astream_invoke("帮我计算12^8+675-233")


2026-04-09 01:46:30,664 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
2026-04-09 01:46:30,666 | INFO | ✅ openairesponses Provider 异步流式工具调用开始


我将用计算工具直接求值，确保结果准确。

2026-04-09 01:46:35,902 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
2026-04-09 01:46:35,903 | INFO | ✅ openairesponses Provider 异步流式工具调用开始


结果是：429982138

In [9]:
test_normal_stream_invoke_with_tool(agent)

2026-04-08 15:33:19,147 | INFO | 对话历史已清空
2026-04-08 15:33:19,148 | WARNING | 工具注册表已存在!
2026-04-08 15:33:19,149 | INFO | 成功添加工具: calculator
2026-04-08 15:33:19,150 | INFO | 使用工具模式流式调用智能体


RuntimeError: Cannot run the event loop while another loop is running